# APIs Lab

In this lab, you will work with different APIs to build datasets and perform basic analysis.

The goal is to practice how to:

- Make API requests  
- Explore JSON responses  
- Extract relevant information  
- Transform data into pandas DataFrames  
- Generate simple insights  

You will work like a data analyst using APIs data.

# Challenge 1: Pokémon Dataset & Analysis

## Goal

Build and analyze a Pokémon dataset using the API.

## Example (single request)

Below you can see how to get data for ONE Pokémon:

In [1]:
import requests

url = "https://pokeapi.co/api/v2/pokemon/1"

response = requests.get(url)
data = response.json()

data
print(data .keys())


dict_keys(['abilities', 'base_experience', 'cries', 'forms', 'game_indices', 'height', 'held_items', 'id', 'is_default', 'location_area_encounters', 'moves', 'name', 'order', 'past_abilities', 'past_stats', 'past_types', 'species', 'sprites', 'stats', 'types', 'weight'])


In [2]:
import requests
import pandas as pd

pokemon_list = []

for i in range(1, 152):
    url = f"https://pokeapi.co/api/v2/pokemon/{i}"
    response = requests.get(url)
    data = response.json()

    name = data["name"]
    height = data["height"]
    weight = data["weight"]
    base_exp = data["base_experience"]
    first_type = data["types"][0]["type"]["name"]

    pokemon_list.append({
        "name": name,
        "height": height,
        "weight": weight,
        "base_experience": base_exp,
        "type": first_type
    })

df = pd.DataFrame(pokemon_list)

# إAdding column for weight in kg
df["weight_kg"] = df["weight"] / 10

# Analysis
heaviest = df.loc[df["weight"].idxmax()]
avg_weight = df["weight"].mean()
type_counts = df["type"].value_counts()
type_avg_weight = df.groupby("type")["weight"].mean()
highest_avg_type = type_avg_weight.idxmax()

print("Heaviest Pokémon:\n", heaviest)
print("\nAverage weight:", avg_weight)
print("\nType counts:\n", type_counts)
print("\nAverage weight per type:\n", type_avg_weight)
print("\nType with highest average weight:", highest_avg_type)

Heaviest Pokémon:
 name               snorlax
height                  21
weight                4600
base_experience        189
type                normal
weight_kg            460.0
Name: 142, dtype: object

Average weight: 459.5165562913907

Type counts:
 type
water       28
normal      22
poison      14
grass       12
fire        12
bug         12
electric     9
rock         9
ground       8
psychic      8
fighting     7
ghost        3
dragon       3
fairy        2
ice          2
Name: count, dtype: int64

Average weight per type:
 type
bug         229.916667
dragon      766.000000
electric    317.888889
fairy       237.500000
fighting    542.857143
fire        480.250000
ghost       135.666667
grass       279.916667
ground      452.625000
ice         480.000000
normal      500.863636
poison      273.142857
psychic     515.625000
rock        876.111111
water       579.678571
Name: weight, dtype: float64

Type with highest average weight: rock


## Tasks

1. Using the example above, get data for Pokémon IDs from 1 to 151

2. Extract:
   - name
   - height
   - weight
   - base_experience
   - first type

3. Create a DataFrame

4. Add a new column:
   - weight_kg (divide weight by 10)

5. Answer:
   - Which Pokémon is the heaviest?
   - What is the average weight?
   - How many Pokémon are there per type?

6. Answer:
   - Which type has the highest average weight?

# Challenge 2: Crypto Market Analysis

## Goal

Analyze cryptocurrency prices using real-time data.

## Tasks

1. Get prices for:
   - bitcoin
   - ethereum
   - solana
   - dogecoin

2. Convert the response into a DataFrame (coins as rows)

3. Rename columns properly

4. Add:
   - price_rank (ranking by price)
   - price_category:
        - "high" if > 1000  
        - "medium" if between 100 and 1000  
        - "low" if < 100  

5. Answer:
   - Which coin is the most expensive?
   - Which category has more coins?

6. **Bonus**:
   - Create a column with price difference vs bitcoin

In [ ]:
# starter code

import requests
import pandas as pd

url = "https://api.coingecko.com/api/v3/simple/price"

params = {
    # complete this
}


In [3]:
import requests
import pandas as pd

url = "https://api.coingecko.com/api/v3/simple/price"

params = {
    "ids": "bitcoin,ethereum,solana,dogecoin",
    "vs_currencies": "usd",
    "include_market_cap": "true",
    "include_24hr_change": "true"
}

response = requests.get(url, params=params)
data = response.json()

df = pd.DataFrame(data).T

df = df.rename(columns={
    "usd": "price_usd",
    "usd_market_cap": "market_cap_usd",
    "usd_24h_change": "change_24h"
})

df["price_rank"] = df["price_usd"].rank(ascending=False, method="dense")

def price_category(price):
    if price > 1000:
        return "high"
    elif price >= 100:
        return "medium"
    else:
        return "low"

df["price_category"] = df["price_usd"].apply(price_category)

most_expensive = df["price_usd"].idxmax()
category_counts = df["price_category"].value_counts()

bitcoin_price = df.loc["bitcoin", "price_usd"]
df["difference_from_bitcoin"] = bitcoin_price - df["price_usd"]

print("Crypto DataFrame:")
print(df)

print("\nMost expensive coin:", most_expensive)
print("\nPrice category counts:")
print(category_counts)

Crypto DataFrame:
             price_usd  market_cap_usd  change_24h  price_rank price_category  \
bitcoin   70921.000000    1.421043e+12   -0.111132         1.0           high   
dogecoin      0.091033    1.400625e+10    0.014667         4.0            low   
ethereum   2186.030000    2.639672e+11   -0.237923         2.0           high   
solana       82.470000    4.736997e+10    0.546561         3.0            low   

          difference_from_bitcoin  
bitcoin                  0.000000  
dogecoin             70920.908967  
ethereum             68734.970000  
solana               70838.530000  

Most expensive coin: bitcoin

Price category counts:
price_category
high    2
low     2
Name: count, dtype: int64
